# Silver Support Tickets

Clean and transform support tickets data from bronze to silver layer with data quality checks: clean data, validate keys, and remove duplicates.

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/CommonHelper"

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/SilverLoader"

In [0]:
silver_table_name = get_table_name("support_tickets", "silver")
bronze_table_name = get_table_name("support_tickets", "bronze")

In [0]:
spark.read.table(bronze_table_name).display()

In [0]:
from pyspark.sql import functions as F

# --- Transform function for silver support_tickets ---
def transform_support_tickets(bronze_df):
    silver_df = (bronze_df
        .select(
            F.col("ticket_id"),
            F.col("customer_id"),
            F.col("order_id"),
            clean_text("issue_category").alias("issue_category"),
            clean_text("subject").alias("subject"),
            clean_text("description").alias("description"),
            clean_text("priority").alias("priority"),
            clean_text("status").alias("status"),
            clean_text("resolution").alias("resolution"),
            F.col("created_at"),
            F.col("resolved_at"),
            F.col("updated_at"),
            F.col("_bronze_ingestion_ts")
        )
    )
    silver_df = validate_keys(silver_df, ["ticket_id"])
    silver_df = remove_duplicates(silver_df, ["ticket_id"], order_col="updated_at")
    silver_df = drop_bronze_metadata(silver_df)
    return silver_df

result = incremental_silver_load(
    bronze_table_name=bronze_table_name,
    silver_table_name=silver_table_name,
    key_columns=["ticket_id"],
    transform_fn=transform_support_tickets,
    update_condition="source.updated_at >= target.updated_at"
)

print(result)

In [0]:
display(spark.read.table(silver_table_name))